In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.stats import norm

In [ ]:
# ─── Paths ────────────────────────────────────────────────────────────────────
BASE     = Path().resolve()
_panels  = sorted(BASE.glob('results_v3/daily_panel_v3_*.parquet'))
PARQUET  = _panels[-1] if _panels else BASE / 'results_v3' / 'daily_panel_v3.parquet'
FEAT_CSV = BASE / 'results_v3' / 'feature_importance.csv'
OUT      = BASE / 'figures'
OUT.mkdir(exist_ok=True)

DPI = 300

# ─── Colour palette ────────────────────────────────────────────────────────────
C1 = '#2c5f8a'   # blue  — Act 1 / general
C2 = '#2d8a4e'   # green — Act 2 / train
C3 = '#7b3d8a'   # purple — Act 3
CR = '#c03530'   # red   — holdout / warning / threshold
CO = '#c97121'   # orange — accent / NaN markers

PREFIX_COL = {
    'recent_':     '#27ae60',
    'cumul_mean_': '#2980b9',
    'cumul_std_':  '#8e44ad',
    'trend_':      '#d35400',
    'delta_':      '#c0392b',
}

# ─── Config (mirrors notebook CONFIG cell) ─────────────────────────────────────
PHYSIOL_COLS = [
    'hr_mean', 'hr_min', 'hr_max', 'rhr_mean',
    'stress_mean', 'resp_mean',
    'body_battery_avg',
    'sleep_awake_s', 'sleep_light_s', 'sleep_deep_s', 'sleep_rem_s',
    'motion_intensity', 'met_avg', 'active_s',
    'baseline_bmi', 'baseline_body_fat_pct', 'baseline_weight_kg',
]
MVPA_THRESHOLD  = 420
MIN_WEAR_PCT    = 60.0
MIN_VALID_DAYS  = 3
RANDOM_SEED     = 42
HOLD_OUT_FRAC   = 0.20

In [3]:
# ─── Helpers ──────────────────────────────────────────────────────────────────

def save(name):
    path = OUT / name
    plt.savefig(path, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f'  ✓  {name}')


def cohens_d(a, b):
    """Pooled-SD Cohen's d (matches notebook cell 28)."""
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    pooled_sd = np.sqrt(
        ((na - 1) * a.std(ddof=1)**2 + (nb - 1) * b.std(ddof=1)**2) / (na + nb - 2)
    )
    return (a.mean() - b.mean()) / pooled_sd if pooled_sd > 0 else 0.0


def icc_one_way(data, group_col, value_col):
    """One-way random-effects ICC(1,1) — matches notebook cell 29."""
    groups = data.dropna(subset=[value_col]).groupby(group_col)[value_col].apply(list)
    groups = [np.array(g) for g in groups if len(g) > 1]
    if len(groups) < 2:
        return np.nan
    k = len(groups)
    n_sizes = [len(g) for g in groups]
    N = sum(n_sizes)
    grand_mean = np.concatenate(groups).mean()
    SSB = sum(n * (g.mean() - grand_mean)**2 for n, g in zip(n_sizes, groups))
    SSW = sum(((g - g.mean())**2).sum() for g in groups)
    MSB = SSB / (k - 1)
    MSW = SSW / (N - k) if N > k else np.nan
    if MSW is None or MSB + MSW == 0:
        return np.nan
    k_star = (N - sum(n**2 for n in n_sizes) / N) / (k - 1)
    icc = (MSB - MSW) / (MSB + (k_star - 1) * MSW)
    return float(np.clip(icc, 0, 1))


def hanley_se(auc, n_pos, n_neg):
    """Hanley & McNeil (1982) SE formula for ROC AUC."""
    Q1  = auc / (2 - auc)
    Q2  = 2 * auc**2 / (1 + auc)
    var = (auc * (1 - auc) + (n_pos - 1) * Q1 + (n_neg - 1) * Q2) / (n_pos * n_neg)
    return np.sqrt(max(var, 1e-10))

In [4]:
# ─── Data loading ─────────────────────────────────────────────────────────────

def load_panel():
    panel = pd.read_parquet(PARQUET)
    panel['date'] = pd.to_datetime(panel['date'])
    return panel


def build_model_df(panel):
    """
    Replicate notebook build_weekly_targets() + cumul_mean_ feature engineering.
    Returns model_df with cumul_mean_{col} columns and next_mvpa_420 target.
    """
    avail = [c for c in PHYSIOL_COLS if c in panel.columns]

    frames = []
    for uid, udf in panel.groupby('user_id'):
        udf = udf.copy().sort_values('date')
        udf['year_week'] = (
            udf['date'].dt.isocalendar().year.astype(str) + '_' +
            udf['date'].dt.isocalendar().week.astype(str).str.zfill(2)
        )
        udf['valid_wear_day'] = (udf['wear_pct'] >= MIN_WEAR_PCT).astype(int)

        wdf = udf.groupby('year_week').agg(
            mvpa_sum   = ('mvpa_min', 'sum'),
            valid_days = ('valid_wear_day', 'sum'),
            **{c: (c, 'mean') for c in avail}
        ).reset_index()

        wdf = wdf[wdf['valid_days'] >= MIN_VALID_DAYS].sort_values('year_week').reset_index(drop=True)
        if len(wdf) < 2:
            continue

        wdf['label_mvpa_420']  = (wdf['mvpa_sum'] >= MVPA_THRESHOLD).astype(int)
        wdf['next_mvpa_420']   = wdf['label_mvpa_420'].shift(-1)
        wdf = wdf.iloc[:-1]   # drop last row (no next-week label)

        # Expanding cumulative means (prior weeks only, shift by 1)
        for c in avail:
            wdf[f'cumul_mean_{c}'] = wdf[c].expanding().mean().shift(1)

        wdf['user_id'] = uid
        frames.append(wdf)

    return pd.concat(frames, ignore_index=True), avail


In [5]:
# ════════════════════════════════════════════════════════════════════════════════
# GROUP A — DIAGRAM FIGURES
# ════════════════════════════════════════════════════════════════════════════════

def fig_three_acts():
    fig, ax = plt.subplots(figsize=(13, 4.2))
    ax.set_xlim(0, 13)
    ax.set_ylim(0, 4.5)
    ax.axis('off')

    acts = [
        ('Act 1', 'Cohort\nCharacterisation',
         'EDA · Signal coverage\nWear compliance · Effect sizes', 2.0, C1),
        ('Act 2', 'Predictive\nModelling',
         'Feature engineering · CV\nClassifier training · Hold-out eval', 6.5, C2),
        ('Act 3', 'Clinical\nValidation',
         'BMI reduction link\nExploratory interpretation', 11.0, C3),
    ]

    BOX_W, BOX_H = 3.2, 3.2

    for act_lbl, title, sub, cx, col in acts:
        rect = mpatches.FancyBboxPatch(
            (cx - BOX_W / 2, 0.6), BOX_W, BOX_H,
            boxstyle='round,pad=0.18',
            linewidth=2.0, edgecolor=col, facecolor=col + '18')
        ax.add_patch(rect)
        ax.text(cx, 3.55, act_lbl, ha='center', va='bottom',
                fontsize=10, fontweight='bold', color=col)
        ax.text(cx, 2.35, title, ha='center', va='center',
                fontsize=12.5, fontweight='bold', color='#1a1a1a',
                multialignment='center')
        ax.text(cx, 0.88, sub, ha='center', va='bottom',
                fontsize=8.5, color='#555', style='italic',
                multialignment='center')

    for x0, x1 in [(3.84, 4.66), (8.34, 9.16)]:
        ax.annotate('', xy=(x1, 2.2), xytext=(x0, 2.2),
                    arrowprops=dict(arrowstyle='->', color='#666',
                                    lw=2.2, mutation_scale=18))

    plt.tight_layout(pad=0.4)
    save('three_acts.png')


def fig_feature_timeline():
    """Gantt-style chart: 5 feature window types on a shared week axis."""
    N_WEEKS = 10

    fig, ax = plt.subplots(figsize=(12, 5.2))

    windows = [
        # (label, start_wk, end_wk, color)
        ('recent_',     7, 8, PREFIX_COL['recent_']),
        ('cumul_mean_', 1, 8, PREFIX_COL['cumul_mean_']),
        ('cumul_std_',  1, 8, PREFIX_COL['cumul_std_']),
        ('trend_',      1, 8, PREFIX_COL['trend_']),
        ('delta_',      1, 8, PREFIX_COL['delta_']),
    ]

    y_ticks = list(range(len(windows)))

    for i, (label, start, end, col) in enumerate(windows):
        ax.barh(i, end - start, left=start, height=0.55,
                color=col, alpha=0.75, edgecolor=col, linewidth=1.2)
        ax.text((start + end) / 2, i, label,
                ha='center', va='center', fontsize=9,
                fontweight='bold', color='white')

    # Prediction week shading and marker
    pred_x = 9
    ax.axvspan(pred_x, pred_x + 1, alpha=0.12, color=CR, zorder=0)
    ax.axvline(pred_x, color=CR, lw=1.8, ls='--', label='Prediction week (k+1)')
    ax.text(pred_x + 0.5, len(windows) - 0.1,
            'Target\nweek\n(k+1)', ha='center', va='top',
            fontsize=8.5, color=CR, fontweight='bold', multialignment='center')

    # Week markers — placed above bars to avoid x-axis overlap
    ax.axvline(8, color='#555', lw=1.2, ls=':', alpha=0.7)
    ax.text(8,   -0.62, 'Week k', ha='center', va='top', fontsize=8.5, color='#555')
    ax.text(1,   -0.62, 'Week 1', ha='center', va='top', fontsize=8.5, color='#555')

    # delta_ footnote — placed inside plot area at bottom-left
    ax.text(0.5, -0.90,
            '* delta_  =  recent_  −  cumul_mean_   (requires both windows to be non-null)',
            ha='center', va='center', fontsize=8, color='#777', style='italic')

    ax.set_yticks(y_ticks)
    ax.set_yticklabels([w[0] for w in windows], fontsize=10)
    ax.set_xlim(0, N_WEEKS + 0.5)
    ax.set_ylim(-1.3, len(windows))
    ax.set_xlabel('Monitoring week index', fontsize=10)
    ax.set_title('Temporal Feature Aggregation Windows Anchored to the Prediction Week',
                 fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.legend(fontsize=9, loc='lower right')
    plt.tight_layout()
    save('feature_timeline.png')


def fig_pipeline():
    steps = [
        (C1, 'Raw Garmin CSV export',
         'Pilot 1 + Pilot 2 combined  ·  35 users  ·  20,440 rows  (1 row = 1 user × 1 day)'),
        (C1, 'build_panel()  —  Valid-wear filter',
         'Wear ≥60% · Study-window clip (days 0–41)  →  17 users  ·  1,183 rows'),
        (C2, 'Longitudinal feature engineering',
         'cumul_mean_  ·  recent_  ·  cumul_std_  ·  trend_  ·  delta_  →  80 features'),
        (C2, 'build_weekly_targets()  —  Label creation',
         'Next-week MVPA ≥420 label  ·  ≥3 valid days / wk  →  12 users  ·  150 user-weeks'),
        (C2, 'Patient-level train / hold-out split  (seed = 42)',
         '9 train patients  (135 wks)            3 hold-out patients  (15 wks)'),
        (C2, 'StratifiedGroupKFold CV  (3-fold)  —  Model training',
         'Logistic Regression  ·  Random Forest  ·  Gradient Boosting  ·  XGBoost'),
        (CR, 'Sealed hold-out evaluation',
         'AUC  ·  AUPRC  ·  Brier score  ·  Expected Calibration Error'),
    ]

    BOX_W = 9.0
    BOX_H = 1.05
    GAP   = 0.5
    CX    = 5.0
    HEIGHT = len(steps) * (BOX_H + GAP) + 0.6

    fig, ax = plt.subplots(figsize=(10, HEIGHT))
    ax.axis('off')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, HEIGHT)

    for i, (col, title, sub) in enumerate(steps):
        y = HEIGHT - (i + 1) * (BOX_H + GAP) + GAP / 2

        rect = mpatches.FancyBboxPatch(
            (CX - BOX_W / 2, y), BOX_W, BOX_H,
            boxstyle='round,pad=0.12',
            linewidth=1.8, edgecolor=col, facecolor=col + '18')
        ax.add_patch(rect)

        ax.text(CX, y + BOX_H * 0.68, title,
                ha='center', va='center', fontsize=9.5,
                fontweight='bold', color='#1a1a1a', multialignment='center')
        ax.text(CX, y + BOX_H * 0.25, sub,
                ha='center', va='center', fontsize=8,
                color='#444', style='italic', multialignment='center')

        if i < len(steps) - 1:
            # xy = arrowhead (top of next box, lower y); xytext = tail (bottom of this box)
            ax.annotate('',
                xy=(CX, y - GAP + 0.08),
                xytext=(CX, y - 0.08),
                arrowprops=dict(arrowstyle='->', color='#666',
                                lw=1.6, mutation_scale=14))

    plt.tight_layout(pad=0.3)
    save('pipeline.png')


def fig_attrition():
    """CONSORT-style attrition flowchart — corrected arrow geometry."""
    fig, ax = plt.subplots(figsize=(9, 11))
    ax.axis('off')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)

    # Layout constants
    BOX_W  = 5.0    # main box width
    BOX_H  = 1.15   # main box height
    CX     = 3.8    # centre x of main column → right edge at 3.8+2.5=6.3
    EX_CX  = 8.4    # centre x of exclusion boxes → left edge at 8.4-1.3=7.1
    EX_W   = 2.6    # exclusion box width
    EX_H   = 0.95   # exclusion box height

    # Main funnel boxes: (y_centre, colour, bold title, italic sub)
    main_boxes = [
        (10.3, C1, 'Raw Garmin export  (Pilot 1 + Pilot 2)',
         'n = 35 users  ·  20,440 rows'),
        (7.6,  C1, 'Clean daily panel',
         'n = 17 users  ·  1,183 rows\n(wear ≥60%, study window)'),
        (4.9,  C2, 'Model-ready dataset',
         'n = 12 users  ·  150 user-weeks\n(≥3 valid days/wk, ≥2 qualifying weeks)'),
    ]

    # Exclusion side-boxes: (y_centre, text)
    excl_data = [
        (8.95, '−18 users excluded\n(no valid wear or\noutside study window)'),
        (6.25, '−5 users excluded\n(insufficient\nqualifying weeks)'),
    ]

    # ── Draw main boxes ─────────────────────────────────────────────────────
    for yc, col, title, sub in main_boxes:
        rect = mpatches.FancyBboxPatch(
            (CX - BOX_W / 2, yc - BOX_H / 2), BOX_W, BOX_H,
            boxstyle='round,pad=0.12',
            linewidth=1.8, edgecolor=col, facecolor=col + '20', zorder=2)
        ax.add_patch(rect)
        n_sub_lines = sub.count('\n') + 1
        title_y = yc + (0.17 if n_sub_lines > 1 else 0.05)
        ax.text(CX, title_y, title,
                ha='center', va='center', fontsize=9.5,
                fontweight='bold', color='#1a1a1a',
                multialignment='center', zorder=3)
        ax.text(CX, yc - 0.22, sub,
                ha='center', va='center', fontsize=8.0,
                color='#444', style='italic',
                multialignment='center', zorder=3)

    # ── Vertical down-arrows + exclusion connectors ──────────────────────────
    for i, ((yc_top, _, _, _), (yc_bot, _, _, _)) in enumerate(
            zip(main_boxes[:-1], main_boxes[1:])):

        tail_y = yc_top - BOX_H / 2   # bottom edge of upper box
        head_y = yc_bot + BOX_H / 2   # top edge of lower box

        # Down arrow: head (xy) at top of lower box, tail (xytext) at bottom of upper box
        # Offsets of 0.17 = 0.12 pad + 0.05 gap to clear the visual box boundary
        ax.annotate('',
            xy=(CX, head_y + 0.17),
            xytext=(CX, tail_y - 0.17),
            arrowprops=dict(arrowstyle='->', color='#555',
                            lw=1.5, mutation_scale=13),
            zorder=1)

        # Exclusion box
        ey_c, excl_txt = excl_data[i]
        ex_left = EX_CX - EX_W / 2
        excl_rect = mpatches.FancyBboxPatch(
            (ex_left, ey_c - EX_H / 2), EX_W, EX_H,
            boxstyle='round,pad=0.10',
            linewidth=1.4, edgecolor=CR, facecolor='#fde8e8', zorder=2)
        ax.add_patch(excl_rect)
        ax.text(EX_CX, ey_c, excl_txt,
                ha='center', va='center', fontsize=7.8,
                color=CR, multialignment='center', zorder=3)

        # Horizontal arrow: tail outside main visual right; head outside excl visual left
        # main pad=0.12 → visual right = main_right+0.12; excl pad=0.10 → visual left = ex_left-0.10
        main_right = CX + BOX_W / 2
        ax.annotate('',
            xy=(ex_left - 0.16, ey_c),
            xytext=(main_right + 0.18, ey_c),
            arrowprops=dict(arrowstyle='->', color=CR,
                            lw=1.2, mutation_scale=11),
            zorder=1)

    # ── Train / Hold-out split ───────────────────────────────────────────────
    # Start vertical line at visual bottom (geometric − pad) so it doesn't thread through the transparent box
    model_bottom = main_boxes[-1][0] - BOX_H / 2 - 0.12   # visual bottom of model-ready box
    junction_y   = 3.0     # raised from 2.4 → longer arrows to split boxes
    train_cx     = 2.1     # centre x of training box
    hold_cx      = 7.2     # centre x of hold-out box
    split_box_w  = 3.5
    split_box_h  = 1.1
    split_box_yc = 1.2     # centre y of both split boxes

    # Vertical line from model-ready box down to junction
    ax.plot([CX, CX], [model_bottom, junction_y],
            color='#555', lw=1.5, zorder=1)

    # Horizontal fork line connecting the two branches
    ax.plot([train_cx, hold_cx], [junction_y, junction_y],
            color='#555', lw=1.5, zorder=1)

    # Vertical arrows from junction DOWN to each split box
    train_box_top = split_box_yc + split_box_h / 2
    hold_box_top  = split_box_yc + split_box_h / 2

    ax.annotate('',
        xy=(train_cx, train_box_top + 0.12),
        xytext=(train_cx, junction_y),
        arrowprops=dict(arrowstyle='->', color='#555',
                        lw=1.5, mutation_scale=13),
        zorder=1)
    ax.annotate('',
        xy=(hold_cx, hold_box_top + 0.12),
        xytext=(hold_cx, junction_y),
        arrowprops=dict(arrowstyle='->', color='#555',
                        lw=1.5, mutation_scale=13),
        zorder=1)

    # Training set box
    tr_rect = mpatches.FancyBboxPatch(
        (train_cx - split_box_w / 2, split_box_yc - split_box_h / 2),
        split_box_w, split_box_h,
        boxstyle='round,pad=0.12',
        linewidth=1.8, edgecolor=C2, facecolor=C2 + '20', zorder=2)
    ax.add_patch(tr_rect)
    ax.text(train_cx, split_box_yc + 0.18, 'Training set  (75%)',
            ha='center', va='center', fontsize=9.5,
            fontweight='bold', color='#1a1a1a', zorder=3)
    ax.text(train_cx, split_box_yc - 0.20, 'n = 9 patients  ·  135 user-weeks',
            ha='center', va='center', fontsize=8,
            color='#444', style='italic', zorder=3)

    # Hold-out box
    ho_rect = mpatches.FancyBboxPatch(
        (hold_cx - split_box_w / 2, split_box_yc - split_box_h / 2),
        split_box_w, split_box_h,
        boxstyle='round,pad=0.12',
        linewidth=1.8, edgecolor=CR, facecolor='#fde8e8', zorder=2)
    ax.add_patch(ho_rect)
    ax.text(hold_cx, split_box_yc + 0.18, 'Hold-out set  (25%)',
            ha='center', va='center', fontsize=9.5,
            fontweight='bold', color=CR, zorder=3)
    ax.text(hold_cx, split_box_yc - 0.20, 'n = 3 patients  ·  15 user-weeks',
            ha='center', va='center', fontsize=8,
            color=CR, style='italic', zorder=3)

    # Split label above junction
    ax.text(CX, junction_y + 0.12, 'Patient-level split  (seed = 42)',
            ha='center', va='bottom', fontsize=7.5, color='#666', style='italic')

    ax.set_title('Patient Attrition Funnel — eBATTLE Pilot 1 & 2',
                 fontsize=12, fontweight='bold', y=0.98)
    plt.tight_layout(pad=0.3)
    save('attrition.png')


In [6]:
# ════════════════════════════════════════════════════════════════════════════════
# GROUP B — PARQUET-DRIVEN FIGURES
# ════════════════════════════════════════════════════════════════════════════════

def fig_cohend(model_df, avail_cols):
    """Cohen's d: active vs inactive cumulative-mean weeks (matches notebook cell 28)."""
    eff = model_df.dropna(subset=['next_mvpa_420']).copy()
    active   = eff[eff['next_mvpa_420'] == 1]
    inactive = eff[eff['next_mvpa_420'] == 0]

    rows = []
    for col in avail_cols:
        cum_col = f'cumul_mean_{col}'
        if cum_col not in eff.columns:
            continue
        a_v = active[cum_col].dropna()
        b_v = inactive[cum_col].dropna()
        if len(a_v) < 5 or len(b_v) < 5:
            continue
        d = cohens_d(a_v, b_v)
        rows.append({'feature': col, 'cohens_d': d, 'abs_d': abs(d)})

    df = pd.DataFrame(rows).sort_values('abs_d', ascending=False).reset_index(drop=True)

    def bar_color(d):
        if abs(d) >= 0.8:
            return CR
        elif abs(d) >= 0.5:
            return CO
        elif abs(d) >= 0.2:
            return C1
        return '#aaaaaa'

    colors = [bar_color(d) for d in df['cohens_d']]

    fig, ax = plt.subplots(figsize=(9, 6.5))
    ax.barh(df['feature'][::-1], df['cohens_d'][::-1],
            color=colors[::-1], edgecolor='white', height=0.68)

    ax.axvline(0,    color='#333',  lw=0.9)
    ax.axvline( 0.8, color=CR,  lw=1.3, ls='--', alpha=0.75, label='Large  |d|≥0.80')
    ax.axvline(-0.8, color=CR,  lw=1.3, ls='--', alpha=0.75)
    ax.axvline( 0.5, color=CO,  lw=1.3, ls='--', alpha=0.75, label='Medium  |d|≥0.50')
    ax.axvline(-0.5, color=CO,  lw=1.3, ls='--', alpha=0.75)

    ax.set_xlabel("Cohen's d  (active weeks − inactive weeks; cumulative-mean signal)",
                  fontsize=10)
    ax.set_title("Effect Size: Active vs Inactive Weeks by Physiological Signal\n"
                 "(cumul_mean_ features; next_mvpa_420 target)",
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(axis='x', alpha=0.35)
    plt.tight_layout()
    save('cohend.png')


def fig_icc(panel):
    """ICC(1,1) per physiological signal (matches notebook cell 29)."""
    rows = []
    for col in PHYSIOL_COLS:
        if col not in panel.columns:
            continue
        icc = icc_one_way(panel, 'user_id', col)
        rows.append({'feature': col, 'icc': icc})

    df = (pd.DataFrame(rows)
          .dropna(subset=['icc'])
          .sort_values('icc', ascending=False)
          .reset_index(drop=True))

    def bar_color(v):
        if v >= 0.75:
            return CR
        elif v >= 0.50:
            return CO
        return C1

    colors = [bar_color(v) for v in df['icc']]

    fig, ax = plt.subplots(figsize=(9, 6.5))
    ax.barh(df['feature'][::-1], df['icc'][::-1],
            color=colors[::-1], edgecolor='white', height=0.68)

    ax.axvline(0.75, color=CR, lw=1.5, ls='--', alpha=0.8,
               label='High ICC ≥0.75  (between-patient dominates)')
    ax.axvline(0.50, color=CO, lw=1.5, ls='--', alpha=0.8,
               label='Moderate ICC ≥0.50')
    ax.set_xlim(0, 1.08)
    ax.set_xlabel('ICC(1,1) — Intraclass Correlation Coefficient', fontsize=10)
    ax.set_title('ICC: Fraction of Variance Attributable to Between-Patient Differences\n'
                 '(High ICC ⟹ model learns patient identity, not weekly patterns)',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.35)
    plt.tight_layout()
    save('icc.png')


In [7]:
# ════════════════════════════════════════════════════════════════════════════════
# GROUP C — CSV / KNOWN-VALUE FIGURES
# ════════════════════════════════════════════════════════════════════════════════

def fig_feat_imp():
    """Top-20 feature importances coloured by prefix type (from feature_importance.csv)."""
    df = pd.read_csv(FEAT_CSV).head(20).sort_values('importance')

    def prefix_color(feat):
        for prefix, col in PREFIX_COL.items():
            if feat.startswith(prefix):
                return col
        return '#aaaaaa'

    colors = [prefix_color(f) for f in df['feature']]

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.barh(df['feature'], df['importance'],
            color=colors, edgecolor='white', height=0.72)

    handles = [mpatches.Patch(facecolor=c, label=p.rstrip('_'))
               for p, c in PREFIX_COL.items()]
    ax.legend(handles=handles, title='Feature type', fontsize=8.5,
              title_fontsize=9, loc='lower right')

    ax.set_xlabel('Mean importance (XGBoost, normalised)', fontsize=10)
    ax.set_title('Top 20 Feature Importances — XGBoost Classifier\n'
                 '(primary target: next_mvpa_420)',
                 fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.35)
    plt.tight_layout()
    save('feat_imp.png')


def fig_power_curve():
    """
    Statistical power vs N patients (Hanley & McNeil 1982).
    Parameters derived from notebook cell 50 output:
      training set: 135 user-weeks, 9 patients  →  weeks_per_pt = 15
      n_pos = 64, n_neg = 71  →  active_frac ≈ 0.474  (matches SE=0.1280 at N=9)
    """
    weeks_per_pt = 15.0
    n_pos_train  = 64          # from back-calculation matching notebook SE=0.1280
    n_neg_train  = 71
    active_frac  = n_pos_train / (n_pos_train + n_neg_train)

    N_range  = np.arange(5, 155, 1)
    powers   = []
    for n_pts in N_range:
        n_wk  = int(n_pts * weeks_per_pt)
        n_pos = max(1, int(n_wk * active_frac))
        n_neg = max(1, n_wk - n_pos)
        se    = hanley_se(0.70, n_pos, n_neg)
        z     = (0.70 - 0.50) / se - norm.ppf(0.95)
        powers.append(norm.cdf(z) * 100)

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.plot(N_range, powers, color=C1, lw=2.5, label='Statistical power')

    ax.axhline(80.0, color=CO, lw=1.5, ls='--', alpha=0.85, label='80% power')
    ax.axhline(88.4, color=CR, lw=1.5, ls='--', alpha=0.85, label='88.4% at N=30')

    ax.axvline(9,  color='#777', lw=1.3, ls=':', alpha=0.85)
    ax.axvline(30, color=CR,    lw=1.5, ls='--', alpha=0.75)

    ax.scatter([9],  [46.7], color='#777', s=80, zorder=6)
    ax.scatter([30], [88.4], color=CR,    s=80, zorder=6)

    ax.annotate('Current\nN=9  (46.7%)', xy=(9, 46.7),
                xytext=(18, 35),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.2),
                fontsize=8.5, color='#555', multialignment='center')
    ax.annotate('N=30  (88.4%)', xy=(30, 88.4),
                xytext=(42, 80),
                arrowprops=dict(arrowstyle='->', color=CR, lw=1.2),
                fontsize=8.5, color=CR)

    ax.set_xlabel('Number of training patients (N)', fontsize=10)
    ax.set_ylabel('Statistical power (%)', fontsize=10)
    ax.set_title('Sample Size vs Statistical Power\n'
                 '(Hanley & McNeil 1982; target AUC = 0.70; α = 0.05, one-sided)',
                 fontsize=11, fontweight='bold')
    ax.set_xlim(5, 150)
    ax.set_ylim(0, 105)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.30)
    plt.tight_layout()
    save('power_curve.png')


def fig_learning_curve():
    """
    CV AUC vs training set size (exact values from notebook cell 50 output).
    Sizes 30 and 42 produced single-class folds (NaN) — marked with dotted lines.
    """
    sizes = [30,   42,   53,    65,    77   ]
    means = [None, None, 0.687, 0.713, 0.692]
    stds  = [None, None, 0.275, 0.260, 0.258]

    valid_x = [s for s, m in zip(sizes, means) if m is not None]
    valid_m = [m for m in means if m is not None]
    valid_s = [s for s, m in zip(stds, means) if m is not None]
    nan_x   = [s for s, m in zip(sizes, means) if m is None]

    fig, ax = plt.subplots(figsize=(8.5, 5.5))

    # Shaded ±1 SD band
    ax.fill_between(valid_x,
                    [m - s for m, s in zip(valid_m, valid_s)],
                    [m + s for m, s in zip(valid_m, valid_s)],
                    alpha=0.15, color=C2, label='±1 SD across folds')

    ax.plot(valid_x, valid_m, 'o-', color=C2,
            lw=2.5, ms=8, zorder=5, label='CV AUC (mean)')
    ax.errorbar(valid_x, valid_m, yerr=valid_s,
                fmt='none', color=C2, capsize=5, alpha=0.6)

    for x in nan_x:
        ax.axvline(x, color='#bbb', lw=1.3, ls=':', alpha=0.9)
        ax.text(x, 0.33, f'{x} rows\nNaN\n(single-class fold)',
                ha='center', va='bottom', fontsize=7.5,
                color='#999', multialignment='center')

    ax.axhline(0.70, color=CR,    lw=1.5, ls='--', alpha=0.85,
               label='Clinical threshold  (AUC = 0.70)')
    ax.axhline(0.50, color='#bbb', lw=1.2, ls='-',  alpha=0.7,
               label='No-information baseline  (AUC = 0.50)')

    # Trend arrow
    ax.annotate('', xy=(valid_x[-1], valid_m[-1]),
                xytext=(valid_x[0], valid_m[0]),
                arrowprops=dict(arrowstyle='->', color=CO, lw=1.8, mutation_scale=12))
    ax.text(63, 0.703, 'Δ AUC = +0.005\n(53 → 77 rows)',
            ha='center', va='bottom', fontsize=8.0, color=CO,
            style='italic', multialignment='center')

    ax.set_xlabel('Training set size (user-weeks)', fontsize=10)
    ax.set_ylabel('Cross-validated AUC  (3-fold stratified group CV)', fontsize=10)
    ax.set_title('Learning Curve — Logistic Regression  ·  Primary Target: next_mvpa_420',
                 fontsize=11, fontweight='bold')
    ax.set_xlim(20, 88)
    ax.set_ylim(0.28, 1.08)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(alpha=0.30)
    plt.tight_layout()
    save('learning_curve.png')



In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    print(f'\nGenerating thesis figures  →  {OUT}')
    print('─' * 62)

    print('\n[Group A]  Diagram figures')
    fig_three_acts()
    fig_feature_timeline()
    fig_pipeline()
    fig_attrition()

    print('\n[Group B]  Parquet-driven figures')
    panel    = load_panel()
    model_df, avail_cols = build_model_df(panel)
    print(f'           model_df: {len(model_df)} rows, {model_df["user_id"].nunique()} users')
    fig_cohend(model_df, avail_cols)
    fig_icc(panel)

    print('\n[Group C]  CSV / known-value figures')
    fig_feat_imp()
    fig_power_curve()
    fig_learning_curve()

    print('\n─' * 62)
    print(f'9 figures saved to {OUT}/')
    print()
    print('Still needed (notebook-dependent):')
    print('  roc.png         — add secondary-target panel to notebook cell 52')
    print('  calibration.png — add reliability diagram to notebook cell 54')
